# TerraWatch M4 ? Learned pipeline demonstration

Use an **L4 GPU**. Run this after obtaining a BIT weights file or training ZIP. The runtime downloads the released RSCaMa (~3.6 GB) and pinned Qwen checkpoint, so allow for cache and network speed. Once the notebook exports a ZIP, disconnect the GPU. This is a two-pair execution sample, with the output decisions recorded exactly as produced. It may yield deferrals on both pairs.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, os, subprocess, sys, shutil, zipfile
import torch
from google.colab import drive, files
assert torch.cuda.is_available(), 'Select an L4 runtime type'
print(torch.cuda.get_device_name(),round(torch.cuda.get_device_properties(0).total_memory/1024**3,1),'GiB')
assert torch.cuda.get_device_properties(0).total_memory>=19*1024**3,'Use an L4 for the unquantised referee'
drive.mount('/content/drive')
# Set one or both paths. If the checkpoint path is empty, an upload prompt opens.
CHECKPOINT_OR_TRAINING_ZIP = ''
LEVIR_ARCHIVE = ''
if not LEVIR_ARCHIVE:
    base=Path('/content/drive/MyDrive')
    matches=sorted({*base.rglob('LEVIR-MCI-dataset.zip'), *base.rglob('levir_mci.zip')})
    assert len(matches)==1, f'Set the exact LEVIR_ARCHIVE path: {matches}'
    LEVIR_ARCHIVE=str(matches[0])
if not CHECKPOINT_OR_TRAINING_ZIP:
    uploads=files.upload()
    assert len(uploads)==1, 'Upload either the training ZIP or the chosen BIT .pt file'
    CHECKPOINT_OR_TRAINING_ZIP=next(iter(uploads))
checkpoint=Path('/content/bit_m4.pt')
selected_input=Path(CHECKPOINT_OR_TRAINING_ZIP)
if selected_input.suffix=='.zip':
    import hashlib
    with zipfile.ZipFile(selected_input) as z:
        receipt=json.loads(z.read('selection.json')); choice=receipt['selected']
        assert choice, 'Use an explicit completed-arm checkpoint when budgets differed'
        name=choice['checkpoint'];data=z.read(name)
        assert hashlib.sha256(data).hexdigest()==receipt['checkpoint_sha256'][name]
        checkpoint.write_bytes(data)
else:
    shutil.copy2(selected_input,checkpoint)


## Prepare isolated worker environments and pinned assets

In [ ]:
repo=Path('/content/terrawatch-m4-demo')
if not repo.exists():
    subprocess.run(['git','clone','--branch','milestone-4-review','https://github.com/shoaib2000857/Group-7-DS-and-AI-Lab-Project.git',str(repo)],check=True,timeout=180)
os.chdir(repo)
subprocess.run(['git','rev-parse','HEAD'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','opencv-python-headless','pydantic>=2,<3','einops'],check=True,timeout=180)
subprocess.run([sys.executable,'scripts/setup_model_sources.py'],check=True,timeout=240)
caption_python='/content/tw-caption-env/bin/python';judge_python='/content/tw-referee-env/bin/python'
for folder in ['/content/tw-caption-env','/content/tw-referee-env']:
    subprocess.run([sys.executable,'-m','venv','--system-site-packages',folder],check=True,timeout=90)
subprocess.run([caption_python,'-m','pip','install','-q','transformers==4.39.3','setuptools<70','wheel','ftfy','regex','einops','tqdm'],check=True,timeout=180)
subprocess.run([caption_python,'-m','pip','install','-q','--no-build-isolation','git+https://github.com/openai/CLIP.git@dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1'],check=True,timeout=180)
subprocess.run([judge_python,'-m','pip','install','-q','transformers==5.17.0','accelerate>=1.1,<2','sentencepiece','safetensors'],check=True,timeout=240)
output=Path('/content')/datetime.now(timezone.utc).strftime('TerraWatch_M4_Learned_%Y%m%d_%H%M%S');output.mkdir()
with zipfile.ZipFile(LEVIR_ARCHIVE) as z:
    captions=output/'LevirCCcaptions.json';captions.write_bytes(z.read('LEVIR-MCI-dataset/LevirCCcaptions.json'))
subprocess.run([sys.executable,'scripts/fetch_model_assets.py','--rscama','--captions',str(captions)],check=True,timeout=900)


## Execute original T1/T2 pairs through the learned workers

In [ ]:
config=json.loads(Path('configs/pipeline.json').read_text())
config['detector'].update(python=sys.executable,checkpoint=str(checkpoint))
config['captioner']['python']=caption_python
config['referee']['python']=judge_python
config['worker_timeout_seconds']=420
config_path=output/'pipeline.json';config_path.write_text(json.dumps(config,indent=2))
subprocess.run([sys.executable,'scripts/check_model_setup.py','--config',str(config_path),'--environments','--strict'],check=True,timeout=150)
for pair in ('train_000010','val_000005'):
    inputs=output/'inputs'/pair;inputs.mkdir(parents=True)
    for date in ('T1','T2'):
        shutil.copy2(repo/'src/app/assets'/f'{pair}_{date}.png',inputs/f'{date}.png')
    case=output/pair
    try:
        subprocess.run([sys.executable,'scripts/run_pipeline.py','--t1',str(inputs/'T1.png'),'--t2',str(inputs/'T2.png'),'--pair-id',pair,'--config',str(config_path),'--output',str(case)],check=True,timeout=720)
        result=json.loads((case/'report.json').read_text());print(pair,result['generated_caption'],[(r['source_span'],r['decision']['outcome']) for r in result['claims']],result['errors'])
        if (case/'detector_response.mask.png').exists():
            for date in ('T1','T2'):shutil.copy2(inputs/f'{date}.png',case/f'{date}.png')
            (case/'review_inputs.json').write_text(json.dumps({'images':['T1.png','T2.png'],'mask':'detector_response.mask.png'}))
    except (subprocess.TimeoutExpired,subprocess.CalledProcessError) as error:
        (output/f'{pair}_execution_error.txt').write_text(str(error));print(pair,error)


## Export reports, masks, crops and worker logs

In [ ]:
captions.unlink(missing_ok=True)
archive=shutil.make_archive(str(output),'zip',output)
files.download(archive)
